# Semana 2 — Joins e Agregações

Combinação de DataFrames com `join` e agregação de dados com `groupBy().agg()`, usando dados sintéticos de pedidos e clientes.

Equivalentes PySpark do `JOIN` e `GROUP BY` do SQL. Parte do plano de reposicionamento para engenharia de dados.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from decimal import Decimal
import random
from datetime import date, timedelta

random.seed(42)
produtos = ["VR", "VA", "VT", "AUXILIO"]
base = date(2026, 1, 1)

schema_pedidos = StructType([
    StructField("id_pedido",    IntegerType(),      False),
    StructField("id_cliente",   IntegerType(),      False),
    StructField("produto",      StringType(),       True),
    StructField("data_pedido",  DateType(),         True),
    StructField("valor_bruto",  DecimalType(12, 2), True),
    StructField("pct_desconto", DecimalType(6, 4),  True),
])

dados = [
    (
        i,
        random.randint(1000, 1050),
        random.choice(produtos),
        base + timedelta(days=random.randint(0, 300)),
        Decimal(str(round(random.uniform(50, 5000), 2))),
        Decimal(str(round(random.uniform(0, 0.15), 4))),
    )
    for i in range(1, 50001)
]

pedidos = spark.createDataFrame(dados, schema_pedidos)
pedidos.show(5)

In [0]:
regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
segmentos = ["Varejo", "Corporativo", "PME"]

schema_clientes = StructType([
    StructField("id_cliente", IntegerType(), False),
    StructField("nome",       StringType(),  True),
    StructField("regiao",     StringType(),  True),
    StructField("segmento",   StringType(),  True),
])

dados_clientes = [
    (
        i,
        f"Cliente {i}",
        random.choice(regioes),
        random.choice(segmentos),
    )
    for i in range(1000, 1046)   # repare: 1000 a 1045, não 1050
]

clientes = spark.createDataFrame(dados_clientes, schema_clientes)
clientes.show(5)
print("Total de clientes:", clientes.count())

In [0]:
pedidos.join(clientes, on="id_cliente", how="inner").show(5)

In [0]:
inner = pedidos.join(clientes, on="id_cliente", how="inner")
print("Pedidos originais:", pedidos.count())
print("Após inner join:  ", inner.count())

In [0]:
left = pedidos.join(clientes, on="id_cliente", how="left")
print("Pedidos originais:", pedidos.count())
print("Após left join:   ", left.count())

In [0]:
orfaos = left.filter(F.col("nome").isNull())
print("Pedidos órfãos:", orfaos.count())
orfaos.select("id_pedido", "id_cliente", "produto", "valor_bruto").show(5)

In [0]:
pedidos.groupBy("produto").agg(F.sum("valor_bruto").alias("receita_total")).show(5)

In [0]:
pedidos.groupBy("produto").agg(
    F.count("*").alias("qtd_pedidos"),
    F.sum("valor_bruto").alias("receita_total"),
    F.avg("valor_bruto").alias("Valor_medio"),
    F.min("valor_bruto").alias("Valor_minimo"),
    F.max("valor_bruto").alias("Valor_maximo"),
    ).show(5)


In [0]:
(
    pedidos
    .join(clientes, on="id_cliente", how="inner")
    .groupBy("regiao")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.sum("valor_bruto").alias("receita_total"),
    )
    .orderBy(F.col("receita_total").desc())
    .show()
)

In [0]:
(
    pedidos
    .join(clientes, on="id_cliente", how="inner")
    .groupBy("segmento")
    .agg(
        F.sum("valor_bruto").alias("receita")
    )
    .orderBy(F.col("receita").desc())
    .show(5)
)

In [0]:
(
    pedidos.join(clientes, on="id_cliente", how="inner")
    .groupBy("regiao", "Produto")
    .agg(
        F.sum("valor_bruto").alias("receita")
    )
    .orderBy(F.col("receita").desc())
    .show(20)
)